# Análisis V3 - Clasificación de Campañas Agrícolas: Regresión Logística y KNN

## 1. Contexto y Justificación del Submuestreo
En las versiones V1 y V2 se evidenció que la proporción desbalanceada (89.8% verano frente a 10.2% invierno) inducía un sesgo severo hacia la clase mayoritaria. KNN en V1 alcanzaba un Accuracy ficticio de 89.35% pero detectaba un único caso de invierno (Recall de apenas 2.38%). En esta versión V3 se evalúan ambos algoritmos sobre el dataset balanceado 50/50 (`0901_Produccion_Arroz_V3_Clasificacion.xlsx`).

## 2. Metodología
* **Datos:** 280 observaciones (140 Verano vs 140 Invierno).
* **Variables:** `SUPERFICIE (Ha)` y `PRODUCCION (qq)` estandarizadas mediante `StandardScaler`.
* **Partición:** 70% entrenamiento y 30% evaluación estratificada (42 muestras por clase en el conjunto de prueba).
* **Optimización:** `GridSearchCV` estratificado (5-fold) evaluando penalización en Logística y número de vecinos ($K$) junto a ponderación por distancia en KNN.

## 3. Resultados y Comparación Histórica

| Algoritmo | Versión | Balance de Datos | Accuracy | Recall (Inv) | Precision (Inv) | F1-Score | ROC-AUC |
|---|---|---|:---:|:---:|:---:|:---:|:---:|
| **KNN** | V1 | Desbalanceado (89.8% Verano) | 89.35% | 2.38% (1/42) | 25.00% | 0.0450 | 0.5120 |
| **KNN** | V2 | Desbalanceado + Distancia | 86.44% | 19.05% (8/42) | 26.67% | 0.2222 | 0.6147 |
| **KNN** | **V3** | **Balanceado 50/50 ($K=5$)** | **60.71%** | **57.14% (24/42)** | **61.54%** | **0.5926** | **0.6842** |
| **Reg. Logística** | V1 | Desbalanceado (Umbral 0.5) | 90.55% | 7.14% (2/28) | 100.0% | 0.1250 | 0.5350 |
| **Reg. Logística** | V2 | Desbalanceado + Balanced | 82.81% | 33.33% (14/42) | 24.56% | 0.2828 | 0.6572 |
| **Reg. Logística** | **V3** | **Balanceado 50/50 ($C=100$)** | **67.86%** | **47.62% (20/42)** | **80.00%** | **0.5970** | **0.7426** |

### Coeficientes Estandarizados de Regresión Logística V3:
* **Superficie (Ha):** $+10.5026$
* **Producción (qq):** $-9.6289$
* **Intercepto:** $+0.4805$

## 4. Interpretación Técnica
1. **Rescate de la sensibilidad en KNN:** El submuestreo equilibrado permitió a KNN pasar de detectar 1 caso en V1 a 24 casos reales en V3 (Recall del 57.14%), demostrando que su deficiencia previa era producto de la distorsión muestral y no de una falla estructural en el principio de distancia.
2. **Alta precisión de la Regresión Logística:** Con un parámetro $C=100$, el modelo alcanzó un 80% de precisión en la clase minoritaria (solo 5 falsos positivos frente a 37 aciertos en verano), consolidando un ROC-AUC de 0.7426.
3. **Lectura Agronómica:** Los coeficientes confirman que una mayor extensión de terreno por unidad producida incrementa la probabilidad logística de pertenecer al ciclo de invierno, caracterizado por menor productividad por hectárea.

## 5. Visualización
![Regresión Logística V3](../03_imagenes/30_logistica_v3.png)
![KNN V3](../03_imagenes/31_knn_v3.png)

## 6. Conclusión y Veredicto
La **Regresión Logística V3** supera a KNN para implementación en producción. Presenta un ROC-AUC superior (0.7426 vs 0.6842), genera menor tasa de falsos positivos y prescinde de retener la matriz completa de entrenamiento en memoria durante las consultas web.